# LSTM Page Replacement — Training

Run this notebook in Google Colab.
Model is mirrored in `ml/model.py` — keep in sync.

## 1. Clone repo

In [ ]:
import os
import subprocess

REPO_URL = "https://github.com/SaMaJiT7/OS_Scheduler"
REPO_DIR = "/content/OS_Scheduler"

if not os.path.exists(REPO_DIR):
    subprocess.run(["git", "clone", REPO_URL, REPO_DIR], check=True)
%cd {REPO_DIR}
print("Repo ready")

## 2. Imports

In [ ]:
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import TensorDataset, DataLoader
from google.colab import files
from ml.vocab import Vocab

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Device: {DEVICE}")

## 3. Upload dataset

In [ ]:
DATASET_FILE = "lstm_ready_dataset.npz"
if not os.path.exists(DATASET_FILE):
    print("Upload lstm_ready_dataset.npz using the file picker:")
    files.upload()

assert os.path.exists(DATASET_FILE), \
    "Upload failed — run Cell 3 again and select the file"

## 4. Load dataset & build vocab

In [ ]:
data = np.load(DATASET_FILE)
X, y = data["X"], data["y"]
print(f"X: {X.shape}, y: {y.shape}")

TOP_K = 50000
UNK_TOKEN = "<unk>"

all_ids = np.concatenate([X.flatten(), y])
ids, counts = np.unique(all_ids, return_counts=True)
top_indices = np.argsort(-counts)[:TOP_K]

id_to_index = {UNK_TOKEN: 0}
index_to_id = {0: UNK_TOKEN}
for i, idx in enumerate(top_indices, start=1):
    pid = int(ids[idx])
    id_to_index[pid] = i
    index_to_id[i] = pid

VOCAB_SIZE = len(id_to_index)
print(f"Vocab size: {VOCAB_SIZE}")

## 5. Convert to indices & split

In [ ]:
max_id = int(all_ids.max())
lookup = np.zeros(max_id + 1, dtype=np.int64)
for pid, idx in id_to_index.items():
    if isinstance(pid, int):
        lookup[pid] = idx
X_idx = lookup[X]
y_idx = lookup[y]

SPLIT = 0.8
split = int(len(X_idx) * SPLIT)
X_train, X_val = X_idx[:split], X_idx[split:]
y_train, y_val = y_idx[:split], y_idx[split:]
print(f"Train: {len(X_train)}, Val: {len(X_val)}")

## 6. DataLoaders

In [ ]:
BATCH_SIZE = 64

train_ds = TensorDataset(torch.tensor(X_train), torch.tensor(y_train))
val_ds = TensorDataset(torch.tensor(X_val), torch.tensor(y_val))
train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True, num_workers=2)
val_loader = DataLoader(val_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=2)

## 7. Define model (mirrors ml/model.py)

In [ ]:
class PageLSTM(nn.Module):
    def __init__(self, vocab_size, embedding_dim=64, hidden_dim=128,
                 num_layers=2, dropout=0.2):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embedding_dim)
        self.lstm = nn.LSTM(
            embedding_dim, hidden_dim, num_layers,
            dropout=dropout, batch_first=True,
        )
        self.fc = nn.Linear(hidden_dim, vocab_size)

    def forward(self, x):
        emb = self.embedding(x)
        lstm_out, _ = self.lstm(emb)
        last_out = lstm_out[:, -1, :]
        logits = self.fc(last_out)
        return logits

model = PageLSTM(VOCAB_SIZE).to(DEVICE)
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)

## 8. Training loop with early stopping

In [ ]:
EPOCHS = 3
PATIENCE = 2
SAVE_PATH = "ml/checkpoints/best_model.pt"

best_val_loss = float("inf")
patience_counter = 0

for epoch in range(1, EPOCHS + 1):
    model.train()
    train_loss = 0
    for batch_X, batch_y in train_loader:
        batch_X, batch_y = batch_X.to(DEVICE), batch_y.to(DEVICE)
        optimizer.zero_grad()
        logits = model(batch_X)
        loss = criterion(logits, batch_y)
        loss.backward()
        optimizer.step()
        train_loss += loss.item()
    train_loss /= len(train_loader)

    model.eval()
    val_loss = 0
    correct = 0
    total = 0
    with torch.no_grad():
        for batch_X, batch_y in val_loader:
            batch_X, batch_y = batch_X.to(DEVICE), batch_y.to(DEVICE)
            logits = model(batch_X)
            loss = criterion(logits, batch_y)
            val_loss += loss.item()
            preds = logits.argmax(dim=1)
            correct += (preds == batch_y).sum().item()
            total += batch_y.size(0)
    val_loss /= len(val_loader)
    val_acc = correct / total

    print(f"Epoch {epoch}: train_loss={train_loss:.4f}, "
          f"val_loss={val_loss:.4f}, val_acc={val_acc:.4f}")

    if val_loss < best_val_loss:
        best_val_loss = val_loss
        patience_counter = 0
        torch.save({
            "epoch": epoch,
            "model_state_dict": model.state_dict(),
            "optimizer_state_dict": optimizer.state_dict(),
            "val_loss": val_loss,
            "val_acc": val_acc,
        }, SAVE_PATH)
        print(f"  -> saved checkpoint")
    else:
        patience_counter += 1
        if patience_counter >= PATIENCE:
            print(f"Early stopping at epoch {epoch}")
            break

print(f"Training complete. Best val_loss: {best_val_loss:.4f}")

## 9. Save vocab and push to GitHub

In [ ]:
vocab = Vocab()
vocab.id_to_index = id_to_index
vocab.index_to_id = index_to_id
vocab.save("vocab.json")
print("Saved vocab.json")

subprocess.run(["git", "add", "-f", "ml/checkpoints/best_model.pt", "vocab.json"], check=True)
subprocess.run(["git", "-c", "user.name=SaMaJiT7",
                "-c", "user.email=sammajit@users.noreply.github.com",
                "commit", "-m", "Phase 3: trained LSTM model + vocab"], check=True)

try:
    from google.colab import userdata
    token = userdata.get("GITHUB_TOKEN")
    if token:
        repo = "SaMaJiT7/OS_Scheduler"
        subprocess.run(["git", "remote", "set-url", "origin",
                        f"https://{token}@github.com/{repo}"], check=True)
    subprocess.run(["git", "push"], check=True)
    print("Pushed to GitHub.")
except Exception as e:
    print(f"Push failed: {e}")
    print("Files saved locally. Push manually:")
    print("  git push")